In [1]:
import os 
os.chdir('../../../../')
os.environ["DPM_TQDM"] = "False"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

!nvidia-smi

Fri Aug 15 08:12:46 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:19:00.0 Off |                  Off |
| 35%   57C    P8             41W /  450W |    5081MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# -*- coding: utf-8 -*-
import os, math, numpy as np, contextlib
from easydict import EasyDict
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from torchvision.models import ResNet50_Weights

# ===============================
# Config
# ===============================
config = EasyDict(
    backbone='DiT',
    train_pt_dir='samplings/dit/train_4.0/dit_train_4.0_1',
    valid_pt_dir='samplings/dit/eval1000_4.0/dit_eval1000_4.0_0',
    batch_size=10, CFG=4.0, epochs=10, val_every=100,
    log_dir="logs/CFG4.0/0815-1:CLIP Training",
    base_lr=1e-3, total_steps=10000, warmup_steps=50, min_lr_ratio=0.10
)
os.makedirs(config.log_dir, exist_ok=True)
writer = SummaryWriter(config.log_dir)

# ===============================
# Model / CLIP
# ===============================
from backbones.dit import DiT
from utils.clip import CLIPEmbedder

model = DiT(trainable=True); model.set_freeze()
device = model.device
clip_model = CLIPEmbedder().to(device)
print(model)

# ===============================
# Dataset / Dataloader
# ===============================
from datasets.pt_dataset import PtDataset
train_loader = DataLoader(PtDataset(config.train_pt_dir), batch_size=config.batch_size, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
valid_loader = DataLoader(PtDataset(config.valid_pt_dir), batch_size=config.batch_size, shuffle=False)
print('dataloaders ready')

# ===============================
# Solver / Optimizer / Scheduler
# ===============================
from solvers.dual.dynamic.gdual_solver_log_deltaL_only import GDual_Solver
from solvers.transforms.loglinear_transform_general import LogLinearTransform
from solvers.param_extractors.table_extractor import Extractor

noise_schedule = model.get_noise_schedule()
solver = GDual_Solver(
    noise_schedule, steps=5, transform=LogLinearTransform(gamma_push=True, gamma_max=3, kappa_max=3, kappa_disable=False),
    param_extractor=Extractor(), skip_type="time_uniform", order=2, use_corrector=False, time_learning=True, train_mode=True
).to(device)
optimizer = torch.optim.AdamW(solver.parameters(), lr=config.base_lr)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda step: 1.0)
print('solver/optimizer')


/home/scpark/miniconda3/envs/rbf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]An error occurred while trying to fetch /data/huggingface/DiT-XL-2-256/transformer: Error no file named diffusion_pytorch_model.safetensors found in directory /data/huggingface/DiT-XL-2-256/transformer.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  67%|██████▋   | 2/3 [00:00<00:00,  5.04it/s]An error occurred while trying to fetch /data/huggingface/DiT-XL-2-256/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /data/huggingface/DiT-XL-2-256/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline co

dataloaders ready
solver/optimizer


In [3]:
# ===============================
# Utils
# ===============================
IMAGENET_CATEGORIES = ResNet50_Weights.DEFAULT.meta["categories"]

def abort_if_bad(tag, value, step=None):
    v = float(value.detach().cpu()) if isinstance(value, torch.Tensor) else float(value)
    if (not math.isfinite(v)) or (v >= 100.0):
        msg = f"[EARLY-STOP] {tag} loss={v:.6f}" + (f" @ step {step}" if step is not None else "")
        print(msg, flush=True); raise RuntimeError(msg)

def save_checkpoint(global_step, save_dir, solver, valid_loss):
    ckpt = {"global_step": int(global_step), "solver_state_dict": solver.state_dict(),
            "valid_loss": float(valid_loss), "config": dict(config)}
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, f"step_{global_step:08d}.pt"); torch.save(ckpt, path); return path

def texts_from_conds(conds):
    if torch.is_tensor(conds): ids = conds.detach().cpu().tolist()
    else: ids = [int(c) for c in conds]
    return [IMAGENET_CATEGORIES[i] if 0 <= int(i) < len(IMAGENET_CATEGORIES) else "object" for i in ids]

def clip_contrastive_loss(images_decoded, texts):
    # Symmetric InfoNCE: CE(image->text) + CE(text->image)
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16) if device.type=='cuda' else contextlib.nullcontext():
        img_emb = clip_model.encode_image(images_decoded)   # [B,D]
        txt_emb = clip_model.encode_text(texts)             # [B,D]
    img_emb = F.normalize(img_emb.float(), dim=-1)
    txt_emb = F.normalize(txt_emb.float(), dim=-1)
    logits = 100.0 * (img_emb @ txt_emb.t())               # [B,B]
    targets = torch.arange(logits.size(0), device=logits.device)
    loss = 0.5 * (F.cross_entropy(logits, targets) + F.cross_entropy(logits.t(), targets))
    with torch.no_grad():
        prob = logits.softmax(dim=-1)
        top1 = (prob.argmax(dim=-1) == targets).float().mean()
        diag = prob[targets, targets].mean()
    return loss, float(top1), float(diag)

def clip_contrastive_loss2(images_decoded, texts):
    # Cosine similarity loss (pairwise diagonal only)
    # loss = 1 - mean( cos(img_i, txt_i) )
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16) if device.type=='cuda' else contextlib.nullcontext():
        img_emb = clip_model.encode_image(images_decoded)   # [B, D]
        txt_emb = clip_model.encode_text(texts)             # [B, D]

    # L2-normalize → cosine
    img_emb = F.normalize(img_emb.float(), dim=-1)
    txt_emb = F.normalize(txt_emb.float(), dim=-1)

    # Cosine similarity matrix
    sim = img_emb @ txt_emb.t()                             # [B, B]
    B = sim.size(0)
    targets = torch.arange(B, device=sim.device)

    # Diagonal (matching pairs)
    diag = sim[targets, targets]                            # [B]
    loss = 1.0 - diag.mean()

    # Metrics for logging (nearest neighbor top-1 by cosine, and mean diag cosine)
    with torch.no_grad():
        top1 = (sim.argmax(dim=-1) == targets).float().mean()
        diag_mean = diag.mean()

    return loss, float(top1), float(diag_mean)
    

# ===============================
# Validation
# ===============================
@torch.no_grad()
def get_valid_loss(device, solver):
    solver.eval()
    psnr_losses, clip_losses, clip_accs, clip_diags = [], [], [], []
    pbar = tqdm(valid_loader, leave=False)
    for batch in pbar:
        noises = batch['noise'].to(device, non_blocking=True)
        conds  = batch['cond']
        targets= batch['sample'].to(device, non_blocking=True)

        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)
        pred_lat = solver.sample(noises, model_fn)

        psnr_loss = torch.log(F.mse_loss(pred_lat, targets) + 1e-8)
        imgs = model.decode_vae(pred_lat, raw_output=True)
        texts = texts_from_conds(conds)
        clip_loss, acc, diag = clip_contrastive_loss(imgs, texts)

        abort_if_bad("valid(batch)", clip_loss)
        psnr_losses.append(psnr_loss.item()); clip_losses.append(clip_loss.item())
        clip_accs.append(acc); clip_diags.append(diag)
        pbar.set_postfix({'val_clip': clip_loss.item(), 'acc': acc})

    vp = float(np.mean(psnr_losses)) if psnr_losses else 0.0
    vc = float(np.mean(clip_losses)) if clip_losses else 0.0
    vacc = float(np.mean(clip_accs)) if clip_accs else 0.0
    vdiag = float(np.mean(clip_diags)) if clip_diags else 0.0
    abort_if_bad("valid(mean)", vc)
    return vp, vc, vacc, vdiag

# ===============================
# Train
# ===============================
def do_train_loop(device, epoch, writer, solver, optimizer, scheduler, global_step_start=0):
    solver.train(); pbar = tqdm(train_loader); losses = []; gstep = global_step_start
    for _, batch in enumerate(pbar):
        if gstep >= config.total_steps: break

        if gstep > 0 and gstep % config.val_every == 0:
            vpsnr, vclip, vacc, vdiag = get_valid_loss(device, solver)
            print(f'step:{gstep} valid_psnr_loss:{vpsnr:.6f}')
            print(f'step:{gstep} valid_clip_loss:{vclip:.6f} (acc={vacc:.3f}, diagP={vdiag:.3f})')
            writer.add_scalar("valid/psnr_loss", vpsnr, gstep)
            writer.add_scalar("valid/clip_loss", vclip, gstep)
            writer.add_scalar("valid/clip_acc",  vacc,  gstep)
            writer.add_scalar("valid/clip_diag_prob", vdiag, gstep)
            save_checkpoint(gstep, config.log_dir, solver, vclip)

        optimizer.zero_grad(set_to_none=True)
        noises = batch['noise'].to(device, non_blocking=True)
        conds  = batch['cond']
        targets= batch['sample'].to(device, non_blocking=True)

        model_fn = model.get_model_fn(noise_schedule, pos_conds=conds, guidance_scale=config.CFG)
        amp = torch.autocast(device_type='cuda', dtype=torch.bfloat16) if device.type=='cuda' else contextlib.nullcontext()
        with amp:
            pred_lat  = solver.sample(noises, model_fn)
            psnr_loss = torch.log(F.mse_loss(pred_lat, targets) + 1e-8)  # proxy
            imgs = model.decode_vae(pred_lat, raw_output=True)
            texts = texts_from_conds(conds)
            clip_loss, acc, diag = clip_contrastive_loss(imgs, texts)
            loss = clip_loss

        abort_if_bad("train", loss, gstep)

        # [GRAD DEBUG] ── (1) backward 직전: 중간 텐서 grad 보존
        pred_lat.retain_grad()
        imgs.retain_grad()

        loss.backward()

        # [GRAD DEBUG] ── (2) backward 직후: grad가 실제로 생겼는지 확인
        lat_g = None if pred_lat.grad is None else pred_lat.grad.norm().item()
        img_g = None if imgs.grad is None else imgs.grad.norm().item()
        tot = sum(1 for p in solver.parameters() if p.requires_grad)
        nz  = sum(1 for p in solver.parameters() if p.grad is not None)
        if gstep % 50 == 0:  # 너무 자주 찍히지 않게
            print(f"[GRAD] lat={lat_g}  img={img_g}  solver params with grad: {nz}/{tot}")

        grad_norm = torch.nn.utils.clip_grad_norm_(solver.parameters(), 1.0)
        if torch.isnan(grad_norm):
            print(f"[SKIP-STEP] non-finite grad_norm={grad_norm.item():.4e}", flush=True)
            optimizer.zero_grad(set_to_none=True); continue

        optimizer.step(); scheduler.step()
        lr_now = optimizer.param_groups[0]["lr"]
        writer.add_scalar("train/lr", lr_now, gstep)
        writer.add_scalar("train/psnr_loss", psnr_loss.item(), gstep)
        writer.add_scalar("train/clip_loss", loss.item(), gstep)
        writer.add_scalar("train/clip_acc",  acc, gstep)
        writer.add_scalar("train/clip_diag_prob", diag, gstep)

        losses.append(loss.item())
        pbar.set_postfix({'loss': loss.item(), 'lr': lr_now, 'acc': acc})
        gstep += 1

    return float(np.mean(losses)) if losses else 0.0, gstep


In [4]:
# ===============================
# Train (minimal main)
# ===============================
def main():
    writer = SummaryWriter(config.log_dir)
    print('tensorboard:', config.log_dir)

    global_step = 0
    for epoch in range(config.epochs):
        if global_step >= config.total_steps:
            break
        mean_loss, global_step = do_train_loop(
            device, epoch, writer, solver, optimizer, scheduler, global_step_start=global_step
        )
        print(f'[epoch {epoch}] mean_train_clip_loss={mean_loss:.6f}, global_step={global_step}')

    # Final validation & checkpoint (CLIP loss)
    val_psnr_mean, val_clip_mean, val_clip_acc, val_clip_diag = get_valid_loss(device, solver)
    save_checkpoint(global_step, config.log_dir, solver, val_clip_mean)
    writer.add_scalar("valid/clip_loss_final", val_clip_mean, global_step)
    writer.add_scalar("valid/psnr_loss_final", val_psnr_mean, global_step)
    writer.close()
    print('done')

if __name__ == "__main__":
    main()


tensorboard: logs/CFG4.0/0815-1:CLIP Training


  0%|          | 1/1000 [00:03<50:27,  3.03s/it, loss=0.503, lr=0.001, acc=0.8]

[GRAD] lat=0.6701083183288574  img=1.6796875  solver params with grad: 2/2


  5%|▌         | 51/1000 [00:58<17:30,  1.11s/it, loss=0.0176, lr=0.001, acc=1] 

[GRAD] lat=0.08256296068429947  img=0.19140625  solver params with grad: 2/2


...

[GRAD] lat=0.03564624860882759  img=0.0537109375  solver params with grad: 2/2


 75%|███████▌  | 751/1000 [22:00<05:57,  1.43s/it, loss=0.237, lr=0.001, acc=0.9]  

[GRAD] lat=0.7893880605697632  img=0.96484375  solver params with grad: 2/2


 80%|████████  | 800/1000 [23:49<05:57,  1.79s/it, loss=0.0263, lr=0.001, acc=1]  


RuntimeError: The size of tensor a (4) must match the size of tensor b (2) at non-singleton dimension 1